# DBS outcome: 3D spectral + clinical attention — verification batteryReproduces the five-test table for the pre-specified model.**Model.** 3D DCT coefficients of the susceptibility-ordered SN/STN volume (20x20x16) and7 clinical covariates each become one scalar token; type embeddings mark imaging vs clinical;one self-attention layer over the joint set; mean-pool to a linear head; plus a linearresidual on the raw concatenated vector.**Tests.**1. family-wise null over the searched grid2. pre-specified cell (select on internal, read external once)3. seed stability over independent initializations4. internal permutation (labels shuffled, full OOF refit)5. imaging ablation (zero / shuffle the volume at test time)Requires a CUDA device and the MSW/CHH caches under `/checkpoints`.

In [ ]:
import sys, os, time, warningswarnings.filterwarnings('ignore')sys.path.insert(0, '/workspace'); sys.path.insert(0, '/workspace/dbs_3d')os.chdir('/workspace')import numpy as np, pandas as pd, torch, torch.nn as nnfrom sklearn.preprocessing import StandardScalerfrom sklearn.model_selection import StratifiedKFoldfrom sklearn.metrics import roc_auc_scoreDEV = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print('device:', DEV)

## Data and modelLoads the 3D volumes, clinical covariates and the `ConcatAttn` definition.

In [ ]:
# everything above `rows=[]` in concat_attn.py is data prep + the model classexec(open('/workspace/dbs_3d/concat_attn.py').read().split('rows=[]')[0])print(f'MSW {VA.shape}  responders {bA.sum()} / non-responders {(bA==0).sum()}')print(f'CHH {VB.shape}  responders {bB.sum()} / non-responders {(bB==0).sum()}')print(f'clinical covariates: {Ca.shape[1]}')

In [ ]:
BASIS, K = '3DDCT', 16          # the pre-specified cellX0, X1 = basis3(VA, BASIS, K), basis3(VB, BASIS, K)T = lambda a: torch.tensor(a, dtype=torch.float32, device=DEV)def gfit(Xtr, Ctr, ytr, Xte, Cte, seed=0, ep=1500):    torch.manual_seed(seed)    m = ConcatAttn(Xtr.shape[1], Ctr.shape[1], typ=True, resid=True).to(DEV)    opt = torch.optim.AdamW(m.parameters(), 3e-3, weight_decay=0.1)    lf = nn.BCEWithLogitsLoss()    xt, ct, yt = T(Xtr), T(Ctr), T(ytr)    for _ in range(ep):        opt.zero_grad(); lf(m(xt, ct), yt).backward(); opt.step()    m.eval()    with torch.no_grad():        return m(T(Xte), T(Cte)).cpu().numpy(), mdef oof_and_external(X0, X1, y, seed=0):    """patient-level OOF on MSW, then one model on all of MSW applied to CHH"""    pr = np.full(len(y), np.nan)    for tr, va in StratifiedKFold(4, shuffle=True, random_state=0).split(X0, y):        s2 = StandardScaler().fit(X0[tr]); s3 = StandardScaler().fit(Ca[tr])        pr[va], _ = gfit(s2.transform(X0[tr]), s3.transform(Ca[tr]), y[tr].astype(float),                         s2.transform(X0[va]), s3.transform(Ca[va]), seed)    s2 = StandardScaler().fit(X0); s3 = StandardScaler().fit(Ca)    pb, m = gfit(s2.transform(X0), s3.transform(Ca), y.astype(float),                 s2.transform(X1), s3.transform(Cb), seed)    return roc_auc_score(y, pr), pb, m

## Test 3 — seed stabilityRun first: it produces the seed-ensembled prediction the other tests use.

In [ ]:
NSEED = 20ints, exts, preds = [], [], []for sd in range(NSEED):    ai, pb, _ = oof_and_external(X0, X1, bA, seed=sd)    ints.append(ai); exts.append(roc_auc_score(bB, pb)); preds.append(pb)    print(f'  seed {sd:2d}  internal {ai:.3f}   external {exts[-1]:.3f}')ints, exts = np.array(ints), np.array(exts)ens = np.mean(preds, 0)print(f'\ninternal {ints.mean():.3f} +- {ints.std():.3f}   (min {ints.min():.3f}, max {ints.max():.3f})')print(f'external {exts.mean():.3f} +- {exts.std():.3f}   (min {exts.min():.3f}, max {exts.max():.3f})')print(f'external below chance in {(exts<0.5).sum()}/{NSEED} seeds')

## Test 2 — pre-specified cellExternal AUC of the seed-ensembled model, with its permutation p and a bootstrap CI.The two disagree by construction at 3 non-responders; report both.

In [ ]:
rng = np.random.RandomState(0)n = len(bB)auc_ens = roc_auc_score(bB, ens)null = np.array([roc_auc_score(bB[rng.permutation(n)], ens) for _ in range(20000)])boot = [roc_auc_score(bB[i], ens[i]) for i in rng.randint(0, n, (20000, n)) if len(np.unique(bB[i])) > 1]print(f'seed-ensembled external AUC {auc_ens:.3f}')print(f'  permutation p        {(null >= auc_ens).mean():.4f}')print(f'  bootstrap 95% CI     [{np.percentile(boot,2.5):.3f}, {np.percentile(boot,97.5):.3f}]')print(f'  null: mean {null.mean():.3f}, SD {null.std():.3f}')

## Test 4 — internal permutationShuffle the MSW labels and refit the **entire** OOF pipeline. Slow: 200 x 5 model fits.

In [ ]:
obs = ints[0]NPERM = 200pnull = []t0 = time.time()for b in range(NPERM):    yp = bA[rng.permutation(len(bA))]    pnull.append(oof_and_external(X0, X1, yp, seed=0)[0])    if (b+1) % 50 == 0:        print(f'  {b+1}/{NPERM}  null mean {np.mean(pnull):.3f}  ({time.time()-t0:.0f}s)')pnull = np.array(pnull)print(f'\nobserved internal {obs:.3f}   null {pnull.mean():.3f} +- {pnull.std():.3f}   p = {(pnull>=obs).mean():.4f}')

## Test 5 — imaging ablationTrain normally, then remove the imaging **only at test time**.`zeroed` hands the model a constant input it can partly ignore; `shuffled` hands it anotherpatient's anatomy. A model that ignores the image is unaffected by both.

In [ ]:
res = {k: [] for k in ['intact','img0','imgshuf','cov0','covshuf']}s2 = StandardScaler().fit(X0); s3 = StandardScaler().fit(Ca)ZB, DB = s2.transform(X1), s3.transform(Cb)for sd in range(5):    _, _, m = gfit(s2.transform(X0), s3.transform(Ca), bA.astype(float), ZB, DB, seed=sd)    r = np.random.RandomState(sd)    ex = lambda Z, D: roc_auc_score(bB, m(T(Z), T(D)).detach().cpu().numpy())    with torch.no_grad():        res['intact'].append(ex(ZB, DB))        res['img0'].append(ex(np.zeros_like(ZB), DB))        res['imgshuf'].append(ex(ZB[r.permutation(len(ZB))], DB))        res['cov0'].append(ex(ZB, np.zeros_like(DB)))        res['covshuf'].append(ex(ZB, DB[r.permutation(len(DB))]))for k, lab in [('intact','intact'),('img0','imaging zeroed'),('imgshuf','imaging shuffled'),               ('cov0','covariates zeroed'),('covshuf','covariates shuffled')]:    v = np.array(res[k]); print(f'  {lab:20s} {v.mean():.3f} +- {v.std():.3f}')d = np.array(res['intact']) - np.array(res['imgshuf'])print(f'\nimaging contribution (intact - shuffled): {d.mean():+.3f} +- {d.std():.3f}')

## Test 1 — family-wise nullRe-scores every cell of the searched grid under shuffled CHH labels and records the**maximum**. Answers: could searching this many cells produce the best result by chance?Uses the stored predictions from `concat_attn.py`.

In [ ]:
import picklefrom scipy.stats import rankdataP = pickle.load(open('/agent-home/runs/concat_attn_preds.pkl','rb'))['PRED']ks = list(P)R = np.array([rankdata(P[k]) for k in ks])n1, n0 = int(bB.sum()), int((bB==0).sum())auc_all = lambda y: (R @ y - n1*(n1+1)/2) / (n1*n0)obs_all = auc_all(bB)mx = np.array([auc_all(bB[rng.permutation(n)]).max() for _ in range(5000)])i = int(obs_all.argmax())print(f'{len(ks)} cells searched')print(f'  best observed      {obs_all[i]:.3f}  ({ks[i]})')print(f'  chance best-of-grid {mx.mean():.3f}  (95th pct {np.percentile(mx,95):.3f})')print(f'  family-wise p      {(mx >= obs_all[i]).mean():.4f}')

## Summary| # | test | what it rules out ||---|---|---|| 1 | family-wise null | the winner being the luckiest of many searched cells || 2 | pre-specified cell | selection on the external set || 3 | seed stability | an initialization lottery || 4 | internal permutation | fitting noise in the training cohort || 5 | imaging ablation | the model ignoring the image and using covariates alone |**Power ceiling.** CHH has 3 non-responders x 34 responders = 102 pairs, so external AUC isquantized in steps of 1/102 and the null has SD ~0.176. p<0.05 requires AUC >= 0.79 no matterhow the model is improved. More seeds, more permutations and bootstrapping refine estimatesbut cannot move this; only more external non-responders can. At the observed effect size,5 non-responders would give p~0.012 and 8 would give p~0.003.